## Inference Lab: Optimal Price
#### Name: Anna Phillips

In [13]:
# code for set up

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

rng = np.random.default_rng(100)

def purchase_probability(price):
    price = np.asarray(price, dtype=float)
    return (
        0.55 * np.exp(-np.logaddexp(0.0, (price - 13.9) / 0.8))
        + 0.30 * np.exp(-np.logaddexp(0.0, (price - 30.9) / 1.2))
        + 0.15 * np.exp(-np.logaddexp(0.0, (price - 58.4) / 1.8))
    )

def sales(price=14.0, rng=None, mean_customers=30):
    price = float(price)
    if not np.isfinite(price):
        raise ValueError("price must be finite")
    if price < 0:
        raise ValueError("price must be nonnegative")
    mean_customers = float(mean_customers)
    if not np.isfinite(mean_customers) or mean_customers < 0:
        raise ValueError("mean_customers must be finite and nonnegative")
    rng = np.random.default_rng(rng)
    customer_count = rng.poisson(mean_customers)
    probability = purchase_probability(price)
    return rng.binomial(1, probability, size=customer_count).astype(np.int8)

In [14]:
# simple grid search to get started

# list of prices to test
prices = np.arange(0, 71, 5)

# set mean customer size
customer_sizes = [100, 200, 500, 1000]

# create empty list for results
grid_results = []

for mean_customers in customer_sizes:

    for price in prices:

        revenue = sales(
            price,
            mean_customers = mean_customers,
            rng = rng
        ) * price

        mean_revenue = revenue.mean()
        customers = len(revenue)

        se = revenue.std(ddof=1) / np.sqrt(customers)

        ci_low = mean_revenue - 1.645 * se
        ci_high = mean_revenue + 1.645 * se

        grid_results.append({
            "mean_customers": mean_customers,
            "price": price,
            "mean_revenue": mean_revenue,
            "se": se,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "customers": customers
        })

grid_df = pd.DataFrame(grid_results)

In [15]:
grid_df.sort_values(
    ["mean_customers", "price"]
)

,mean_customers,price,mean_revenue,se,ci_low,ci_high,customers
0,100,0,0.000000,0.000000,0.000000,0.000000,111
1,100,5,5.000000,0.000000,5.000000,5.000000,108
2,100,10,10.000000,0.000000,10.000000,10.000000,101
3,100,15,7.568807,0.721657,6.381681,8.755934,109
4,100,20,9.387755,1.013441,7.720644,11.054866,98
5,100,25,10.396040,1.232166,8.369126,12.422953,101
6,100,30,10.000000,1.348400,7.781882,12.218118,111
7,100,35,5.894737,1.350996,3.672348,8.117125,95
8,100,40,6.000000,1.309307,3.846189,8.153811,120
9,100,45,5.416667,1.415567,3.088059,7.745274,108


In [16]:
grid_df.sort_values(["mean_customers", "mean_revenue"], ascending=[True, False]).groupby("mean_customers").head(3)

,mean_customers,price,mean_revenue,se,ci_low,ci_high,customers
10,100,50,12.105263,2.209086,8.471317,15.739210,95
5,100,25,10.396040,1.232166,8.369126,12.422953,101
2,100,10,10.000000,0.000000,10.000000,10.000000,101
21,200,30,11.465969,1.057581,9.726248,13.205689,191
20,200,25,11.312849,0.932680,9.778591,12.847107,179
17,200,10,9.943503,0.056497,9.850565,10.036441,177
35,500,25,12.074148,0.559813,11.153256,12.995041,499
32,500,10,9.978118,0.021882,9.942123,10.014114,457
36,500,30,9.550781,0.618226,8.533799,10.567763,512
50,1000,25,11.654527,0.402512,10.992395,12.316659,961
